# Colab 06 (Compare): RL GRPO vs REINFORCE (Inline)
- `colab_05_rl_full.ipynb` 기반 비교 실험 노트북
- 동일 seed / 동일 eval set에서 `grpo`와 `reinforce`를 비교
- 출력: per-problem 비교 CSV + 알고리즘 요약 CSV


In [ ]:
!pip -q install -U unsloth tqdm
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets accelerate bitsandbytes trl peft


## 1) 설정


In [ ]:
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.optim import AdamW
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder
from tqdm import trange
from unsloth import FastLanguageModel

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 40000,
    'model_type': 'llama8b',
    'model_path': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',  # policy adapter only
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # RL loop
    'epochs': 1,
    'episodes_per_epoch': 2,
    'learning_rate': 5e-5,
    'temperature': 0.7,
    'top_p': 0.9,
    'top_k': 40,
    'max_new_tokens': 1,  # legacy field; action decoding uses step_action_max_new_tokens

    'baseline_beta': 0.9,
    'use_running_baseline': False,

    # algo
    'rl_algo': 'grpo',  # single run fallback (reinforce | grpo | bopo)

    # compare mode
    'compare_algorithms': ['grpo', 'reinforce', 'bopo'],
    'compare_output_dir': '/content/outputs/jssp_rl_compare',
    'compare_eval_source': 'hf',  # same_as_train | hf | local
    'compare_eval_hf_repo': 'HYUNJINI/jssp_validation_all_v1',
    'compare_eval_hf_file': 'validation_data/ta.json',
    'compare_eval_local_path': '/content/ta.json',
    'compare_eval_instances': 20,
    'compare_eval_num_return_sequences': 1,
    'compare_eval_enable_step_improvement': False,
    'group_size': 4,
    'grpo_epochs': 2,
    'clip_epsilon': 0.2,
    'kl_coef': 0.0,
    'bopo_beta': 2.0,
    'bopo_gap_scale': 3.0,
    'bopo_margin': 0.0,
    'bopo_min_relative_gap': 0.0,
    'bopo_max_pairs_per_group': 256,
    'bopo_max_step_pairs_per_pair': 32,
    'bopo_pair_mode': 'shared_prefix',
    'reward_mode': 'raw_neg_makespan',
    'rl_update_mode': 'adapter_only',

    # step decoding/prompt
    'step_action_max_new_tokens': 8,
    'disable_step_problem_context': False,
    'enable_step_improvement': False,
    'step_reflection_passes': 1,
    'step_reflection_topk': 3,
    'invalid_makespan_penalty': 1e6,
    'disable_masking': False,
    'action_code_width': 4,
    'action_code_seed': 42,
    'action_code_cap': 9999,
    'print_step_trace': False,

    # data source
    'use_random_problems': False,
    'dataset_source': 'hf',  # hf | local
    'dataset_hf_repo': 'HYUNJINI/jssp_raw_source_v1',
    'dataset_hf_file': 'llm_jssp/train.json',
    'dataset_local_path': '/content/train.json',
    'dataset_path': None,
    'hf_token': HF_TOKEN,

    # random problem settings
    'random_jobs': 10,
    'random_machines': 10,
    'random_time_low': 1,
    'random_time_high': 100,

    'seed': 42,

    # save/upload
    'output_dir': '/content/outputs/jssp_rl_full',
    'compare_keep_checkpoints': False,
    'save_every_epoch': True,
    'upload_after_train': False,
    'hf_model_repo_out': 'HYUNJINI/RL_jssp_policy_llama8b_compare_full_v1',
    'upload_source': 'latest_checkpoint',
    'checkpoint_tag': 'checkpoint-epoch-1',
}

BOPO_PRESETS = {
    'fast_stable': {
        'disable_masking': False,
        'step_action_max_new_tokens': 8,
        'temperature': 0.7,
        'top_p': 0.9,
        'top_k': 40,
        'group_size': 3,
        'bopo_beta': 1.5,
        'bopo_gap_scale': 2.0,
        'bopo_margin': 0.0,
        'bopo_min_relative_gap': 0.01,
        'bopo_max_pairs_per_group': 64,
        'bopo_max_step_pairs_per_pair': 8,
        'enable_step_improvement': False,
        'print_step_trace': False,
    },
    'high_performance': {
        'disable_masking': False,
        'step_action_max_new_tokens': 8,
        'temperature': 0.8,
        'top_p': 0.92,
        'top_k': 40,
        'group_size': 6,
        'bopo_beta': 2.5,
        'bopo_gap_scale': 4.0,
        'bopo_margin': 0.0,
        'bopo_min_relative_gap': 0.0,
        'bopo_max_pairs_per_group': 192,
        'bopo_max_step_pairs_per_pair': 24,
        'enable_step_improvement': False,
        'print_step_trace': False,
    },
}

# choose one: fast_stable | high_performance
CFG['bopo_preset'] = 'fast_stable'
CFG.update(BOPO_PRESETS[CFG['bopo_preset']])

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    'qwen2.5_7b': 'unsloth/Qwen2.5-7B-Instruct-unsloth-bnb-4bit',
    'qwen2.5_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
}

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print(CFG)


## 2) 유틸/환경/GRPO 함수 (inline)


In [ ]:
import math

import os

import random

import re

import time

from functools import lru_cache

from dataclasses import dataclass

from pathlib import Path

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

import torch

import torch.nn.functional as F

from torch.optim import AdamW

from tqdm import tqdm, trange

def parse_matrix_form_jssp(matrix_content: str, sep: str = ' '):
    lines = [ln.strip() for ln in matrix_content.strip().splitlines() if ln.strip()]
    n, m = map(int, lines[0].split(sep))
    inst_for_ortools = []
    for i in range(n):
        nums = list(map(int, lines[i + 1].split(sep)))
        if len(nums) != 2 * m:
            raise ValueError(f'row {i} has {len(nums)} ints, expected {2*m}')
        inst_for_ortools.append([[nums[2 * j], nums[2 * j + 1]] for j in range(m)])
    ms = None
    if len(lines) >= n + 2:
        try:
            ms = float(lines[n + 1])
        except Exception:
            ms = None
    return n, m, inst_for_ortools, ms

def generate_random_instance(num_jobs=10, num_machines=10, process_time_range=(1, 100), rng=None):
    if rng is None:
        rng = np.random.default_rng()

    inst_for_ortools = []
    for _ in range(num_jobs):
        machines = rng.permutation(num_machines).tolist()
        durations = rng.integers(process_time_range[0], process_time_range[1] + 1, size=num_machines).tolist()
        inst_for_ortools.append([[int(m), int(t)] for m, t in zip(machines, durations)])

    return {'inst_for_ortools': inst_for_ortools}

ACTION_TOKEN_PREFIX = "A"

LEGACY_ACTION_TOKEN_PREFIX = "S"

ACTION_CODE_PATTERN = re.compile(
    r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>",
    re.IGNORECASE,
)

def format_action_code(
    code_index: int,
    code_width: int = 4,
    prefix: str = ACTION_TOKEN_PREFIX,
) -> str:
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"

def parse_action_code(text: str, code_width: int = 4) -> Optional[str]:
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)

class StepActionParseError(ValueError):
    """Raised when a generated step action cannot be parsed."""

def _normalize_text(text: str) -> str:
    return text.replace("\r", "")

def _is_prefix_of_any(candidate_text: str, feasible_action_codes: Sequence[str]) -> bool:
    candidate_text = _normalize_text(candidate_text)
    for code in feasible_action_codes:
        if code.startswith(candidate_text):
            return True
    return False

def build_step_prefix_allowed_tokens_fn(
    tokenizer,
    feasible_action_codes_provider: Iterable[str],
    prompt_len: int = 0,
    code_width: int = 4,
    code_cap: int = 9999,
):
    feasible_action_codes = tuple(str(code) for code in feasible_action_codes_provider)
    if not feasible_action_codes:
        raise RuntimeError("No feasible action codes available at this step.")

    feasible_token_ids = []
    for action_code in feasible_action_codes:
        token_id = tokenizer.convert_tokens_to_ids(str(action_code))
        if token_id is None:
            raise RuntimeError(f"Tokenizer returned None token id for action_code={action_code!r}")
        token_id = int(token_id)
        unk_id = getattr(tokenizer, 'unk_token_id', None)
        if unk_id is not None and token_id == int(unk_id):
            encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
            if len(encoded) != 1:
                raise RuntimeError(
                    f"Action code {action_code!r} is not a single tokenizer token."
                )
            token_id = int(encoded[0])
        feasible_token_ids.append(token_id)
    feasible_token_ids = tuple(sorted(set(feasible_token_ids)))
    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    def prefix_allowed_tokens_fn(batch_id: int, input_ids) -> List[int]:
        if hasattr(input_ids, "tolist"):
            input_ids = input_ids.tolist()
        generated_ids = input_ids[int(prompt_len) :]
        if len(generated_ids) == 0:
            return list(feasible_token_ids)

        chosen_action_code = token_id_to_action_code(
            tokenizer,
            int(generated_ids[0]),
            code_width=code_width,
        )
        if (
            len(generated_ids) == 1
            and chosen_action_code is not None
            and str(chosen_action_code) in feasible_action_codes
            and eos_token_id is not None
        ):
            return [int(eos_token_id)]

        if len(generated_ids) == 1 and chosen_action_code is not None and str(chosen_action_code) in feasible_action_codes:
            return list(feasible_token_ids)

        generated_text = _normalize_text(
            tokenizer.decode(generated_ids, skip_special_tokens=False)
        )
        raise RuntimeError(
            "No valid next tokens for step action generation. "
            f"generated_text={generated_text!r}, feasible_action_codes={list(feasible_action_codes)}"
        )

    return prefix_allowed_tokens_fn

HEADER_PATTERN = re.compile(
    r"JSSP\s+with\s+(\d+)\s+Jobs,\s*(\d+)\s+Machines",
    re.IGNORECASE,
)

JOB_HEADER_PATTERN = re.compile(
    r"^\s*Job\s+(\d+)\s+consists\s+of\s+Operations:\s*$",
    re.IGNORECASE,
)

OPERATION_PATTERN = re.compile(
    r"^\s*Operation\s+(\d+):\s*M(\d+),\s*(\d+)\s*$",
    re.IGNORECASE,
)

SOLUTION_OPERATION_PATTERN = re.compile(
    r"Job\s*(\d+)\s*Operation\s*(\d+),\s*M(\d+)",
    re.IGNORECASE,
)

MAKESPAN_PATTERN = re.compile(r"Makespan\s*:\s*(\d+)", re.IGNORECASE)

@dataclass(frozen=True)
class ParsedTeacherAction:
    """Parsed action from one-shot output text."""

    job_id: int
    op_idx: int
    machine_id: int

def parse_prompt_jobs_first(prompt_text: str, strict: bool = True) -> List[List[List[int]]]:
    """
    Parse `prompt_jobs_first` text into `inst_for_ortools` format.

    Returns:
        inst_for_ortools[job][op] = [machine, duration]
    """
    if not isinstance(prompt_text, str) or not prompt_text.strip():
        raise ValueError("prompt_text must be a non-empty string.")

    header_match = HEADER_PATTERN.search(prompt_text)
    expected_jobs = int(header_match.group(1)) if header_match else None
    expected_machines = int(header_match.group(2)) if header_match else None

    jobs: Dict[int, List[List[int]]] = {}
    current_job: Optional[int] = None

    for raw_line in prompt_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        job_match = JOB_HEADER_PATTERN.match(line)
        if job_match:
            current_job = int(job_match.group(1))
            jobs.setdefault(current_job, [])
            continue

        op_match = OPERATION_PATTERN.match(line)
        if op_match:
            if current_job is None:
                raise ValueError(f"Operation line appears before job header: {raw_line!r}")

            op_idx = int(op_match.group(1))
            machine = int(op_match.group(2))
            duration = int(op_match.group(3))
            expected_op_idx = len(jobs[current_job])
            if strict and op_idx != expected_op_idx:
                raise ValueError(
                    f"Operation index mismatch in Job {current_job}: "
                    f"expected {expected_op_idx}, got {op_idx}."
                )
            jobs[current_job].append([machine, duration])

    if not jobs:
        raise ValueError("Failed to parse any jobs from prompt_text.")

    max_job_id = max(jobs.keys())
    if strict and set(jobs.keys()) != set(range(max_job_id + 1)):
        raise ValueError("Job IDs must be contiguous from 0 when strict=True.")

    inst_for_ortools = [jobs[job_id] for job_id in range(max_job_id + 1)]

    if strict:
        if expected_jobs is not None and expected_jobs != len(inst_for_ortools):
            raise ValueError(
                f"Header jobs ({expected_jobs}) != parsed jobs ({len(inst_for_ortools)})."
            )
        if expected_machines is not None:
            for job_id, job_ops in enumerate(inst_for_ortools):
                if len(job_ops) != expected_machines:
                    raise ValueError(
                        f"Job {job_id} has {len(job_ops)} ops, expected {expected_machines}."
                    )

    return inst_for_ortools

def parse_solution_actions(
    solution_text: str, strict: bool = True
) -> Tuple[List[ParsedTeacherAction], Optional[int]]:
    """
    Parse one-shot output into per-step teacher actions.

    Returns:
        actions in decode order, declared makespan from text if present.
    """
    if not isinstance(solution_text, str) or not solution_text.strip():
        raise ValueError("solution_text must be a non-empty string.")

    matches = SOLUTION_OPERATION_PATTERN.findall(solution_text)
    if not matches:
        raise ValueError("No 'Job X Operation Y, MZ' actions found in solution_text.")

    next_expected_op: Dict[int, int] = {}
    actions: List[ParsedTeacherAction] = []
    for job_str, op_str, machine_str in matches:
        job_id = int(job_str)
        op_idx = int(op_str)
        machine_id = int(machine_str)

        expected_op_idx = next_expected_op.get(job_id, 0)
        if strict and op_idx != expected_op_idx:
            raise ValueError(
                f"Teacher action order violation for job {job_id}: "
                f"expected op {expected_op_idx}, got {op_idx}."
            )

        next_expected_op[job_id] = op_idx + 1
        actions.append(ParsedTeacherAction(job_id=job_id, op_idx=op_idx, machine_id=machine_id))

    makespan_match = MAKESPAN_PATTERN.search(solution_text)
    declared_makespan = int(makespan_match.group(1)) if makespan_match else None
    return actions, declared_makespan

class StaticJSSPStepEnv:
    """
    Deterministic static JSSP environment for step-by-step action selection.

    Action:
        choose one feasible job id at each step.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.scheduled_ops = 0
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(cls, prompt_text: str, strict: bool = True) -> "StaticJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.scheduled_ops = 0
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def get_feasible_jobs(self) -> List[int]:
        return [
            job_id
            for job_id in range(self.num_jobs)
            if self.job_next_op[job_id] < self.operations_per_job[job_id]
        ]

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines

        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1

        return machine_remaining_load, machine_remaining_ops

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(
                float(total_ops - rem_ops) / float(total_ops)
            )
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")

        op_idx = self.job_next_op[job_id]
        if op_idx >= self.operations_per_job[job_id]:
            raise ValueError(f"Job {job_id} has no remaining operation.")

        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = max(self.job_ready_time[job_id], self.machine_ready_time[machine_id])
        end_time = start_time + duration

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": job_id,
            "op_idx": op_idx,
            "machine_id": machine_id,
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        """
        Roll out a full teacher sequence and collect step-level records.
        """
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return list(self.event_log)

def _format_machine_ready(machine_ready_time: Sequence[int]) -> str:
    return " ".join(f"M{i}={t}" for i, t in enumerate(machine_ready_time))

def _format_feasible_jobs(feasible_jobs: Sequence[int]) -> str:
    if not feasible_jobs:
        return "[]"
    return "[" + ", ".join(f"Job {j}" for j in feasible_jobs) + "]"

def _format_action_codes(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_route_tokens(route_tokens: Sequence[str]) -> str:
    if not route_tokens:
        return "[]"
    return "[" + ", ".join(str(token) for token in route_tokens) + "]"

def _safe_ratio(numerator: float, denominator: float) -> float:
    if float(denominator) == 0.0:
        return 0.0
    return float(numerator) / float(denominator)

def _format_value(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)

def _machine_token(machine_id: int) -> str:
    return f"M{int(machine_id)}" if int(machine_id) >= 0 else "M-"

def build_randomized_action_code_map(
    feasible_jobs: Sequence[int],
    rng: Optional[random.Random] = None,
    code_width: int = 4,
    code_start: int = 1,
    code_cap: int = 9999,
) -> Dict[str, int]:
    """
    Create randomized action-code mapping for one step.

    Example output:
        {"<A0102>": 7, "<A4377>": 3, ...}
    """
    jobs = [int(j) for j in feasible_jobs]
    if len(set(jobs)) != len(jobs):
        raise ValueError(f"feasible_jobs contains duplicates: {jobs}")

    if code_start < 0:
        raise ValueError(f"code_start must be non-negative, got {code_start}")
    if code_cap < code_start:
        raise ValueError(
            f"code_cap must be >= code_start, got code_start={code_start}, code_cap={code_cap}"
        )
    if len(jobs) > (int(code_cap) - int(code_start) + 1):
        raise ValueError(
            "Not enough action-code slots for this step: "
            f"jobs={len(jobs)}, available={int(code_cap) - int(code_start) + 1}, "
            f"range=[{code_start}, {code_cap}]"
        )

    if not jobs:
        return {}

    shuffled_jobs = list(jobs)
    if rng is None:
        random.shuffle(shuffled_jobs)
        sampled_code_indices = random.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )
    else:
        rng.shuffle(shuffled_jobs)
        sampled_code_indices = rng.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )

    action_code_to_job: Dict[str, int] = {}
    for code_idx, job_id in zip(sampled_code_indices, shuffled_jobs):
        code = format_action_code(int(code_idx), code_width=code_width)
        action_code_to_job[code] = int(job_id)
    return action_code_to_job

def invert_action_code_map(action_code_to_job: Dict[str, int]) -> Dict[int, str]:
    job_to_action_code: Dict[int, str] = {}
    for code, job_id in action_code_to_job.items():
        if job_id in job_to_action_code:
            raise ValueError(f"Duplicate job id in action_code_to_job: {job_id}")
        job_to_action_code[int(job_id)] = str(code)
    return job_to_action_code

def build_problem_context_text(inst_for_ortools: Sequence[Sequence[Sequence[int]]]) -> str:
    """
    Build minimal static problem context text.
    """
    num_jobs = len(inst_for_ortools)
    total_ops = sum(len(job_ops) for job_ops in inst_for_ortools)
    max_machine = -1
    for job_ops in inst_for_ortools:
        for machine_id, _ in job_ops:
            max_machine = max(max_machine, int(machine_id))
    num_machines = max_machine + 1 if max_machine >= 0 else 0

    return f"Problem: {num_jobs} jobs x {num_machines} machines (total_ops={total_ops})"

def summarize_global_dynamic_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(state_json.get("remaining_work", [])))  # type: ignore[arg-type]
    )
    summary = {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_cmax": current_cmax,
        "total_remaining_work": total_remaining_work,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }
    return summary

def compute_action_transition_features(
    state_json: Dict[str, object],
    action_code_to_job: Dict[str, int],
) -> Tuple[int, List[Dict[str, object]]]:
    job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
    next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
    next2_machine: List[int] = state_json.get("next2_machine", [-1] * len(next_machine))  # type: ignore[assignment]
    next2_proc_time: List[int] = state_json.get("next2_proc_time", [0] * len(next_machine))  # type: ignore[assignment]
    remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
    remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
    job_total_ops: List[int] = state_json.get("job_total_ops", [1] * len(next_machine))  # type: ignore[assignment]
    job_total_work: List[int] = state_json.get("job_total_work", [1] * len(next_machine))  # type: ignore[assignment]
    machine_remaining_load: List[int] = state_json.get("machine_remaining_load", [0] * len(machine_ready_time))  # type: ignore[assignment]
    machine_remaining_ops: List[int] = state_json.get("machine_remaining_ops", [0] * len(machine_ready_time))  # type: ignore[assignment]
    post_route_tokens_by_job: List[List[str]] = state_json.get("post_route_tokens", [[] for _ in next_machine])  # type: ignore[assignment]

    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(int(x) for x in remaining_work))
    )

    effects: List[Dict[str, object]] = []
    for action_code, job_id in action_code_to_job.items():
        job = int(job_id)
        machine_id = int(next_machine[job])
        proc_time = int(next_proc_time[job])
        if machine_id < 0 or proc_time <= 0:
            continue

        job_ready_before = int(job_ready_time[job])
        machine_ready_before = int(machine_ready_time[machine_id])
        est_start = max(job_ready_before, machine_ready_before)
        est_end = est_start + proc_time
        current_cmax_after = max(current_cmax, est_end)
        delta_cmax = current_cmax_after - current_cmax
        rem_ops_before = int(remaining_ops[job])
        rem_ops_after = max(0, rem_ops_before - 1)
        rem_work_before = int(remaining_work[job])
        rem_work_after = max(0, rem_work_before - proc_time)
        total_ops = max(int(job_total_ops[job]), 1)
        total_work = max(int(job_total_work[job]), 1)
        job_progress_ratio_before = float(total_ops - rem_ops_before) / float(total_ops)
        job_progress_ratio_after = float(total_ops - rem_ops_after) / float(total_ops)
        affected_machine_load = int(machine_remaining_load[machine_id])
        affected_machine_ops_left = int(machine_remaining_ops[machine_id])

        effect = {
            "action_code": str(action_code),
            "job_id": job,
            "machine_id": machine_id,
            "machine_token": _machine_token(machine_id),
            "proc_time": proc_time,
            "next_machine": machine_id,
            "next_proc_time": proc_time,
            "next2_machine": int(next2_machine[job]),
            "next2_proc_time": int(next2_proc_time[job]),
            "remaining_ops_before": rem_ops_before,
            "remaining_ops_after": rem_ops_after,
            "remaining_work_before": rem_work_before,
            "remaining_work_after": rem_work_after,
            "job_progress_ratio_before": float(job_progress_ratio_before),
            "job_progress_ratio_after": float(job_progress_ratio_after),
            "job_ready_before": job_ready_before,
            "job_ready_after": int(est_end),
            "machine_ready_before": machine_ready_before,
            "machine_ready_after": int(est_end),
            "estimated_start": int(est_start),
            "estimated_end": int(est_end),
            "est_start": int(est_start),
            "est_end": int(est_end),
            "current_cmax_before": int(current_cmax),
            "current_cmax_after": int(current_cmax_after),
            "estimated_makespan_after": int(current_cmax_after),
            "delta_cmax": int(delta_cmax),
            "delta_cmax_ratio": float(_safe_ratio(delta_cmax, max(current_cmax_after, 1))),
            "job_wait": int(max(0, machine_ready_before - job_ready_before)),
            "machine_idle_gap": int(max(0, job_ready_before - machine_ready_before)),
            "slack_to_current_cmax": int(current_cmax - est_end),
            "affected_machine_load": affected_machine_load,
            "affected_machine_ops_left": affected_machine_ops_left,
            "affected_machine_load_ratio": float(
                _safe_ratio(affected_machine_load, max(total_remaining_work, 1))
            ),
            "remaining_work_after_ratio": float(_safe_ratio(rem_work_after, total_work)),
            "post_route_tokens": list(post_route_tokens_by_job[job]),
            "post_route_len": int(len(post_route_tokens_by_job[job])),
        }
        effects.append(effect)

    effects.sort(
        key=lambda x: (
            int(x["estimated_makespan_after"]),
            int(x["estimated_start"]),
            int(x["proc_time"]),
            int(x["job_id"]),
        )
    )
    return int(current_cmax), effects

def render_action_transition_line(effect: Dict[str, object]) -> str:
    return (
        f"{effect['action_code']} | "
        f"operation machine={_machine_token(int(effect['next_machine']))} | "
        f"processing time={effect['next_proc_time']} | "
        f"rem_ops:{effect['remaining_ops_before']}->{effect['remaining_ops_after']} | "
        f"rem_work:{effect['remaining_work_before']}->{effect['remaining_work_after']} | "
        f"job_prog:{_format_value(effect['job_progress_ratio_before'])}->{_format_value(effect['job_progress_ratio_after'])} | "
        f"job_ready:{effect['job_ready_before']}->{effect['job_ready_after']} | "
        f"machine_ready:{effect['machine_ready_before']}->{effect['machine_ready_after']} | "
        f"est_start={effect['estimated_start']} | "
        f"est_end={effect['estimated_end']} | "
        f"cmax:{effect['current_cmax_before']}->{effect['current_cmax_after']} | "
        f"delta_cmax={effect['delta_cmax']} | "
        f"delta_cmax_ratio={_format_value(effect['delta_cmax_ratio'])} | "
        f"job_wait={effect['job_wait']} | "
        f"machine_idle_gap={effect['machine_idle_gap']} | "
        f"slack_to_cmax={effect['slack_to_current_cmax']} | "
        f"machine_load={effect['affected_machine_load']} | "
        f"machine_ops_left={effect['affected_machine_ops_left']} | "
        f"machine_load_ratio={_format_value(effect['affected_machine_load_ratio'])} | "
        f"rem_work_after_ratio={_format_value(effect['remaining_work_after_ratio'])} | "
        f"post_route={_format_route_tokens(effect.get('post_route_tokens', []))}"
    )

def build_step_prompt(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    step_idx: int,
    total_steps: int,
    problem_context_text: Optional[str] = None,
    action_code_to_job: Optional[Dict[str, int]] = None,
) -> str:
    """
    Build deterministic compact prompt for one-step action selection.

    Output format is always one line.
    - Default (legacy): Action: Job <id>
    - Action-token mode: <Axxxx>
    """
    lines = [
        "You are solving JSSP step-by-step.",
        "Objective: Minimize final makespan (Cmax) while keeping every action feasible.",
    ]
    if problem_context_text:
        lines.extend(
            [
                "Static problem context:",
                problem_context_text,
            ]
        )

    global_summary = summarize_global_dynamic_state(state_json)
    lines.extend(
        [
            "Global dynamic state:",
            (
                f"step={int(global_summary['step_idx']) + 1}/{int(global_summary['total_steps'])} "
                f"scheduled_ratio={_format_value(global_summary['scheduled_ratio'])}"
            ),
            f"current_cmax={global_summary['current_cmax']}",
            f"total_remaining_work={global_summary['total_remaining_work']}",
            (
                f"unfinished_jobs_count={global_summary['unfinished_jobs_count']} "
                f"unfinished_jobs_ratio={_format_value(global_summary['unfinished_jobs_ratio'])}"
            ),
            (
                f"machine_ready_min={global_summary['machine_ready_min']} "
                f"machine_ready_mean={_format_value(global_summary['machine_ready_mean'])} "
                f"machine_ready_max={global_summary['machine_ready_max']} "
                f"machine_ready_std={_format_value(global_summary['machine_ready_std'])}"
            ),
            (
                f"bottleneck_machine={_machine_token(int(global_summary['bottleneck_machine_id']))} "
                f"bottleneck_load={global_summary['bottleneck_machine_load']} "
                f"bottleneck_ops_left={global_summary['bottleneck_machine_ops_left']}"
            ),
        ]
    )

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        lines.extend(
            [
                f"Feasible action codes: {_format_action_codes(action_codes)}",
                "Candidate transition features:",
            ]
        )
        for effect in action_effects:
            lines.append(render_action_transition_line(effect))
        lines.extend(
            [
                "Action codes are randomized at each step. Do not assume persistent code identity.",
                "Choose exactly one feasible action code.",
                "Return exactly one code from the feasible action set, for example <A3812>.",
            ]
        )
    else:
        job_next_op: List[int] = state_json["job_next_op"]  # type: ignore[assignment]
        next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
        next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
        job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
        remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
        remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
        machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
        lines.extend(
            [
                f"Feasible jobs: {_format_feasible_jobs(feasible_jobs)}",
                "Job state summary:",
            ]
        )
        for job_id in range(len(job_next_op)):
            lines.append(
                (
                    f"Job {job_id}: "
                    f"next_op={job_next_op[job_id]}, "
                    f"next_machine={next_machine[job_id]}, "
                    f"next_proc={next_proc_time[job_id]}, "
                    f"job_ready={job_ready_time[job_id]}, "
                    f"remaining_ops={remaining_ops[job_id]}, "
                    f"remaining_work={remaining_work[job_id]}"
                )
            )
        lines.extend(
            [
                f"Machine ready times: {_format_machine_ready(machine_ready_time)}",
                "Choose exactly one job from feasible jobs to minimize final makespan (Cmax).",
                "Return exactly one line: Action: Job <id>",
            ]
        )

    return "\n".join(lines)

def build_step_improvement_prompt(
    state_text: str,
    candidate_action_text: str,
    feasible_jobs: Sequence[object],
    reflection_memory: Optional[str] = None,
    step_diagnostics: Optional[str] = None,
) -> str:
    """Build a hindsight-aware step-improvement prompt."""
    feasible_str: str
    if feasible_jobs and isinstance(feasible_jobs[0], str):
        feasible_str = _format_action_codes([str(x) for x in feasible_jobs])
        output_format = "<Axxxx>"
    else:
        feasible_str = _format_feasible_jobs([int(x) for x in feasible_jobs])
        output_format = "Action: Job <id>"
    prompt = (
        "You are revising one decision inside a completed JSSP schedule.\n"
        "Use hindsight from the full episode, not only local greedy signals.\n"
        "If a small short-term sacrifice helps earlier bottleneck activation, lower idle loss, "
        "or better downstream route progression, prefer it.\n\n"
        "Current step state:\n"
        f"{state_text}\n\n"
        "Previously chosen action in the failed/improvable rollout:\n"
        f"{candidate_action_text}\n\n"
        "Rules:\n"
        "- Objective: minimize final makespan (Cmax).\n"
        "- Think in hindsight: ask which choice would have reduced downstream bottleneck delay, idle gaps, or route blocking.\n"
        "- Prefer the action that best aligns with bottleneck-machine usage, downstream route progression, and lower regret against strong alternatives.\n"
        f"- Final action must be one of feasible options: {feasible_str}\n"
        f"- Return exactly one output in this format: {output_format}\n"
        "- Do not output explanation.\n"
        "- Do not repeat the previous action if the reflection evidence says an alternative is structurally better.\n"
    )
    if reflection_memory:
        prompt += f"\nEpisode hindsight reflection:\n{reflection_memory}\n"
    if step_diagnostics:
        prompt += f"\nCritical-step diagnostics:\n{step_diagnostics}\n"
    return prompt

def build_step_rationale_prompt(
    state_text: str,
    chosen_job: Optional[int] = None,
    feasible_jobs: Optional[Sequence[int]] = None,
    chosen_action_code: Optional[str] = None,
    feasible_action_codes: Optional[Sequence[str]] = None,
) -> str:
    """
    Build explanation prompt after action selection.

    This prompt is intentionally separated from action decoding so that
    feasibility/format masking of action is unaffected.
    """
    using_codes = chosen_action_code is not None
    if using_codes:
        chosen_label = str(chosen_action_code)
        if feasible_action_codes is None:
            raise ValueError("feasible_action_codes is required when chosen_action_code is used.")
        other_labels = [str(code) for code in feasible_action_codes if str(code) != chosen_label]
        others = ", ".join(other_labels) if other_labels else "(none)"
    else:
        if chosen_job is None or feasible_jobs is None:
            raise ValueError("chosen_job and feasible_jobs are required in legacy rationale mode.")
        chosen_label = f"Job {int(chosen_job)}"
        other_jobs = [int(j) for j in feasible_jobs if int(j) != int(chosen_job)]
        others = ", ".join(f"Job {j}" for j in other_jobs) if other_jobs else "(none)"

    return (
        "Explain why the already-selected action is reasonable for this JSSP step.\n"
        "Focus on final makespan (Cmax) and feasibility.\n\n"
        f"{state_text}\n\n"
        f"Selected action (fixed, do not change): {chosen_label}\n"
        f"Other feasible options: {others}\n\n"
        "Output format:\n"
        "Reason: <one concise sentence>\n"
        "Rules:\n"
        "- Output exactly one line starting with 'Reason:'.\n"
        "- Keep it concise (<= 25 words).\n"
        "- Do not change the selected action.\n"
        "- Do not output any 'Action:' line.\n"
        "- Do not output 'Not chosen:'.\n"
    )

@dataclass
class EpisodeBatch:
    log_probs: torch.Tensor
    advantages: torch.Tensor

@dataclass
class TrajectorySample:
    sequence_ids: torch.Tensor
    prompt_len: int
    reward: float
    advantage: float
    old_log_prob: torch.Tensor
    feasible: bool
    makespan: float

@dataclass
class BOPOStepPair:
    winner_sequence_ids: torch.Tensor
    winner_prompt_len: int
    loser_sequence_ids: torch.Tensor
    loser_prompt_len: int
    relative_gap: float
    winner_makespan: float
    loser_makespan: float

@dataclass
class StepActionTrace:
    sequence_ids: torch.Tensor
    prompt_len: int
    chosen_job: int
    step_idx: int

class ExponentialBaseline:
    """Simple running baseline for REINFORCE style updates."""

    def __init__(self, beta: float = 0.9):
        self.beta = beta
        self.value: Optional[float] = None

    def update(self, reward: float) -> float:
        if self.value is None:
            self.value = reward
        else:
            self.value = self.beta * self.value + (1 - self.beta) * reward
        return self.value

def mwkr_schedule(inst_for_ortools: List[List[List[int]]]) -> Tuple[List[Dict], float]:
    """Most Work Remaining heuristic schedule."""

    num_jobs = len(inst_for_ortools)
    num_machines = len(inst_for_ortools[0])

    job_next = [0] * num_jobs
    job_time = [0.0] * num_jobs
    machine_time = [0.0] * num_machines
    total_remaining = [
        sum(op[1] for op in job_ops) for job_ops in inst_for_ortools
    ]

    schedule = []

    while any(idx < len(inst_for_ortools[j]) for j, idx in enumerate(job_next)):
        # Jobs ready to schedule are those with remaining operations
        ready_jobs = [
            j for j in range(num_jobs) if job_next[j] < len(inst_for_ortools[j])
        ]
        if not ready_jobs:
            break

        # Choose job with maximum remaining work (MWKR)
        job = max(ready_jobs, key=lambda j: total_remaining[j])
        op_idx = job_next[job]
        machine, duration = inst_for_ortools[job][op_idx]

        start = max(job_time[job], machine_time[machine])
        end = start + duration

        schedule.append(
            {
                "Job": job,
                "Operation": op_idx,
                "Machine": machine,
                "Start Time": start,
                "Duration": duration,
                "End Time": end,
            }
        )

        job_next[job] += 1
        job_time[job] = end
        machine_time[machine] = end
        total_remaining[job] -= duration

    makespan = max(job_time) if schedule else float("inf")
    return schedule, makespan

def _count_total_ops(inst_for_ortools: List[List[List[int]]]) -> int:
    return int(sum(len(job_ops) for job_ops in inst_for_ortools))


def compute_episode_reward(
    makespan: float,
    feasible: bool,
    inst_for_ortools: List[List[List[int]]],
    invalid_makespan_penalty: float,
    reward_mode: str = "raw_neg_makespan",
    heuristic_makespan: float | None = None,
) -> float:
    if not feasible or not math.isfinite(float(makespan)):
        if reward_mode == "neg_makespan_per_op":
            total_ops = max(_count_total_ops(inst_for_ortools), 1)
            return -float(invalid_makespan_penalty) / float(total_ops)
        if reward_mode == "mwkr_relative":
            return -1.0
        return -float(invalid_makespan_penalty)

    makespan = float(makespan)
    if reward_mode == "raw_neg_makespan":
        return -makespan
    if reward_mode == "neg_makespan_per_op":
        total_ops = max(_count_total_ops(inst_for_ortools), 1)
        return -(makespan / float(total_ops))
    if reward_mode == "mwkr_relative":
        baseline_ms = float(heuristic_makespan) if heuristic_makespan is not None else float(mwkr_schedule(inst_for_ortools)[1])
        if not math.isfinite(baseline_ms) or baseline_ms <= 0:
            return -makespan
        return (baseline_ms - makespan) / baseline_ms
    raise ValueError(f"Unsupported reward_mode={reward_mode}")

def _build_step_chat_prompt(tokenizer, state_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert JSSP scheduler. "
                "Primary objective: minimize final makespan (Cmax). "
                "Choose exactly one feasible action code for this step. "
                "Output exactly one code such as <A3812> and nothing else."
            ),
        },
        {"role": "user", "content": state_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def _build_step_improvement_chat_prompt(tokenizer, improvement_prompt_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are refining one JSSP step action. "
                "Primary objective: minimize final makespan (Cmax). "
                "Return exactly one feasible action code such as <A3812>."
            ),
        },
        {"role": "user", "content": improvement_prompt_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_step_action(
    model,
    tokenizer,
    prompt_text: str,
    feasible_action_codes: List[str],
    device: torch.device,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
    top_k: int = 50,
    use_masking: bool = True,
    action_code_width: int = 4,
    action_code_cap: int = 9999,
) -> Tuple[str, torch.Tensor, int, str]:
    if not feasible_action_codes:
        raise StepActionParseError("No feasible action codes available at this step.")

    prompt_inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    prompt_len = int(prompt_inputs["input_ids"].size(1))

    was_training = model.training
    model.eval()
    try:
        generation_kwargs = dict(
            **prompt_inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=float(temperature),
            top_p=float(top_p),
            top_k=int(top_k),
            return_dict_in_generate=False,
        )

        if use_masking:
            generation_kwargs["prefix_allowed_tokens_fn"] = build_step_prefix_allowed_tokens_fn(
                tokenizer=tokenizer,
                feasible_action_codes_provider=list(feasible_action_codes),
                prompt_len=prompt_len,
                code_width=action_code_width,
                code_cap=action_code_cap,
            )

        with torch.no_grad():
            generated = model.generate(**generation_kwargs)

        sequence_ids = generated[0].detach().cpu()
        new_ids = sequence_ids[prompt_len:]
        if new_ids.numel() <= 0:
            raise StepActionParseError("No action code was generated.")

        generated_text = tokenizer.decode(new_ids, skip_special_tokens=False).strip()
        chosen_action_code = parse_action_code(generated_text, code_width=action_code_width)
        if chosen_action_code is None:
            raise StepActionParseError(
                f"Failed to parse generated text into action code. output={generated_text!r}"
            )
        if str(chosen_action_code) not in feasible_action_codes:
            raise StepActionParseError(
                f"Generated action code is not feasible. parsed={chosen_action_code}, feasible={list(feasible_action_codes)}"
            )
        return str(chosen_action_code), sequence_ids, prompt_len, str(chosen_action_code)
    finally:
        if was_training:
            model.train()

def _estimate_action_effects(state_json, action_code_to_job):
    return compute_action_transition_features(state_json, action_code_to_job)

def _best_alternative_option(step):
    chosen_code = str(step.get("chosen_action_code"))
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    if not alternatives:
        return None
    return min(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
            int(x.get("proc_time", 10**12)),
        ),
    )

def _critical_step_score(step):
    best_alt = _best_alternative_option(step)
    if best_alt is None:
        return None
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    best_alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
    immediate_gap = int(chosen_ms - best_alt_ms)
    makespan_jump = int(step.get("makespan_after", 0) - step.get("makespan_before", 0))
    chosen_start = int(step.get("chosen_start_time", 0))
    best_alt_start = int(best_alt.get("estimated_start", chosen_start))
    start_delay = int(chosen_start - best_alt_start)
    if immediate_gap <= 0 and makespan_jump <= 0 and start_delay <= 0:
        return None
    return (immediate_gap, makespan_jump, start_delay, int(step.get("step_idx", -1)))

def _select_top_critical_steps(step_records, top_k: int = 3):
    scored = []
    for step in step_records:
        score = _critical_step_score(step)
        if score is None:
            continue
        scored.append((score, step))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [step for _, step in scored[: max(1, int(top_k))]]

def _select_critical_step(step_records):
    top = _select_top_critical_steps(step_records, top_k=1)
    return top[0] if top else None

def _build_step_diagnostics(step):
    chosen_code = str(step.get("chosen_action_code"))
    chosen_job = int(step.get("chosen_job", -1))
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    chosen_start = int(step.get("chosen_start_time", 0))
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    alternatives = sorted(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
        ),
    )
    lines = [
        (
            f"Chosen={chosen_code}/Job{chosen_job}, est_Cmax_after={chosen_ms}, "
            f"start={chosen_start}, machine=M{int(step.get('machine_id', -1))}"
        )
    ]
    for rank, opt in enumerate(alternatives[:3], start=1):
        lines.append(
            (
                f"Alt{rank}={opt['action_code']}/Job{int(opt['job_id'])}, "
                f"est_Cmax_after={int(opt['estimated_makespan_after'])}, "
                f"est_start={int(opt['estimated_start'])}, "
                f"proc={int(opt['proc_time'])}"
            )
        )
    return "\n".join(lines)

def _build_reflection_memory(current_makespan: float, critical_steps):
    lines = [
        f"Current episode makespan={float(current_makespan):.1f}.",
        "Reflection rules (avoid bottleneck choices, prefer lower estimated Cmax/start):",
    ]
    for rank, step in enumerate(critical_steps, start=1):
        best_alt = _best_alternative_option(step)
        if best_alt is None:
            continue
        chosen_code = str(step.get("chosen_action_code"))
        chosen_job = int(step.get("chosen_job", -1))
        chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
        chosen_start = int(step.get("chosen_start_time", 0))
        alt_code = str(best_alt.get("action_code"))
        alt_job = int(best_alt.get("job_id", -1))
        alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
        alt_start = int(best_alt.get("estimated_start", chosen_start))
        lines.append(
            (
                f"Rule {rank}: step {int(step.get('step_idx', -1))}, avoid {chosen_code}/Job{chosen_job} "
                f"(est_Cmax={chosen_ms}, start={chosen_start}); "
                f"prefer {alt_code}/Job{alt_job} (est_Cmax={alt_ms}, start={alt_start})."
            )
        )
    return "\n".join(lines)

def _print_step_trace(step_records):
    for step in step_records:
        print(
            f"[step {int(step['step_idx']):03d}] ms {int(step['makespan_before'])}->{int(step['makespan_after'])} "
            f"chosen={step['chosen_action_code']}->Job{step['chosen_job']} "
            f"(M{step['machine_id']},t={step['chosen_proc_time']},"
            f"start={step['chosen_start_time']},end={step['chosen_end_time']})"
        )
        not_chosen = step.get("not_chosen_options", [])
        if not_chosen:
            parts = []
            for opt in not_chosen:
                parts.append(
                    f"{opt['action_code']}->Job{opt['job_id']}"
                    f"(M{opt['machine_id']},t={opt['proc_time']},ms={opt['estimated_makespan_after']},"
                    f"dC={opt['delta_cmax']})"
                )
            print("  not_chosen:", "; ".join(parts))

def rollout_step_episode(
    model,
    tokenizer,
    inst_for_ortools: List[List[List[int]]],
    device: torch.device,
    step_action_max_new_tokens: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
    top_k: int = 50,
    use_masking: bool = True,
    include_problem_context: bool = True,
    enable_step_improvement: bool = False,
    step_reflection_passes: int = 1,
    step_reflection_topk: int = 3,
    action_code_width: int = 4,
    action_code_seed: int = 42,
    action_code_cap: int = 9999,
    print_step_trace: bool = False,
) -> Tuple[float, bool, List[StepActionTrace]]:
    problem_context_text = (
        build_problem_context_text(inst_for_ortools) if include_problem_context else None
    )

    def _rollout_once(code_seed: int, guidance_by_step=None, reflection_memory_text: str | None = None):
        env = StaticJSSPStepEnv(inst_for_ortools)
        env.reset()
        traces: List[StepActionTrace] = []
        step_records = []
        action_rng = random.Random(int(code_seed))
        guidance_map = guidance_by_step or {}

        try:
            while not env.is_done():
                state = env.get_state_json()
                feasible_jobs = [int(j) for j in state["feasible_jobs"]]
                step_idx = int(state["step_idx"])
                action_code_to_job = build_randomized_action_code_map(
                    feasible_jobs=feasible_jobs,
                    rng=action_rng,
                    code_width=action_code_width,
                    code_start=1,
                    code_cap=action_code_cap,
                )
                makespan_before, action_effects = _estimate_action_effects(
                    state_json=state,
                    action_code_to_job=action_code_to_job,
                )
                effect_by_code = {x["action_code"]: x for x in action_effects}
                feasible_action_codes = list(action_code_to_job.keys())
                state_text = build_step_prompt(
                    state_json=state,
                    feasible_jobs=feasible_jobs,
                    step_idx=step_idx,
                    total_steps=int(state["total_steps"]),
                    problem_context_text=problem_context_text,
                    action_code_to_job=action_code_to_job,
                )
                if reflection_memory_text:
                    state_text = (
                        f"{state_text}\n"
                        f"Episode reflection memory:\n{reflection_memory_text}\n"
                        "Apply these reflection rules while selecting this step action."
                    )
                if step_idx in guidance_map:
                    step_guidance = dict(guidance_map[step_idx])
                    preferred_job = int(step_guidance.get("preferred_job", -1))
                    avoid_jobs = [
                        int(x) for x in step_guidance.get("avoid_jobs", [])
                        if int(x) >= 0
                    ]
                    reason_text = str(step_guidance.get("reason", "")).strip()
                    guide_lines = [
                        "Post-episode guidance: This step was identified as a Cmax bottleneck.",
                    ]
                    if preferred_job >= 0:
                        guide_lines.append(f"If feasible, prefer Job {preferred_job}.")
                    if avoid_jobs:
                        guide_lines.append(
                            "Avoid these jobs if strong alternatives exist: "
                            + ", ".join(f"Job {j}" for j in avoid_jobs)
                        )
                    if reason_text:
                        guide_lines.append(f"Why: {reason_text}")
                    state_text = f"{state_text}\n" + "\n".join(guide_lines)

                base_prompt = _build_step_chat_prompt(tokenizer, state_text)
                chosen_action_code, sequence_ids, prompt_len, _ = generate_step_action(
                    model=model,
                    tokenizer=tokenizer,
                    prompt_text=base_prompt,
                    feasible_action_codes=feasible_action_codes,
                    device=device,
                    max_new_tokens=step_action_max_new_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    top_k=top_k,
                    use_masking=use_masking,
                    action_code_width=action_code_width,
                    action_code_cap=action_code_cap,
                )
                chosen_job = int(action_code_to_job[chosen_action_code])
                chosen_effect = effect_by_code.get(chosen_action_code)
                if chosen_effect is None:
                    raise RuntimeError(
                        "Internal mismatch: chosen action code is missing in step option effects. "
                        f"chosen_action_code={chosen_action_code}, available={list(effect_by_code.keys())}"
                    )

                _, _, _, info = env.step(chosen_job)
                traces.append(
                    StepActionTrace(
                        sequence_ids=sequence_ids,
                        prompt_len=prompt_len,
                        chosen_job=chosen_job,
                        step_idx=step_idx,
                    )
                )
                step_records.append(
                    {
                        "step_idx": int(step_idx),
                        "state_text": state_text,
                        "feasible_action_codes": feasible_action_codes,
                        "action_code_to_job": action_code_to_job,
                        "chosen_action_code": str(chosen_action_code),
                        "chosen_job": int(chosen_job),
                        "machine_id": int(info["machine_id"]),
                        "chosen_start_time": int(info["start_time"]),
                        "chosen_end_time": int(info["end_time"]),
                        "chosen_proc_time": int(info["duration"]),
                        "makespan_before": int(makespan_before),
                        "makespan_after": int(info["makespan_so_far"]),
                        "chosen_estimated_makespan_after": int(chosen_effect["estimated_makespan_after"]),
                        "all_options": action_effects,
                        "not_chosen_options": [
                            x for x in action_effects
                            if str(x["action_code"]) != str(chosen_action_code)
                        ],
                        "guidance_applied": bool(step_idx in guidance_map),
                    }
                )
        except StepActionParseError:
            raise
        except Exception as exc:
            raise RuntimeError(
                f"Unexpected rollout error at step {step_idx}: {exc}"
            ) from exc

        return float(env.get_makespan()), True, traces, step_records, []

    makespan, feasible, traces, step_records, notes = _rollout_once(
        code_seed=int(action_code_seed),
        guidance_by_step=None,
        reflection_memory_text=None,
    )
    if not feasible:
        return makespan, feasible, traces

    best_makespan = float(makespan)
    best_traces = traces
    best_step_records = step_records
    improvement_notes = list(notes)

    if enable_step_improvement and int(step_reflection_passes) > 0:
        if print_step_trace:
            print(
                f"[Episode Improvement] start: passes={int(step_reflection_passes)}, "
                f"topk={max(1, int(step_reflection_topk))}, "
                f"baseline_makespan={float(best_makespan):.1f}"
            )
        for pass_idx in range(int(step_reflection_passes)):
            critical_steps = _select_top_critical_steps(
                best_step_records,
                top_k=max(1, int(step_reflection_topk)),
            )
            if not critical_steps:
                improvement_notes.append(f"pass {pass_idx + 1}: no critical step found")
                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                        "no critical step found"
                    )
                break

            reflection_memory = _build_reflection_memory(best_makespan, critical_steps)
            guidance_map = {}
            for critical_step in critical_steps:
                step_idx = int(critical_step["step_idx"])
                feasible_action_codes = list(critical_step["feasible_action_codes"])
                action_code_to_job = dict(critical_step["action_code_to_job"])
                chosen_code = str(critical_step["chosen_action_code"])
                chosen_job = int(critical_step["chosen_job"])
                best_alt = _best_alternative_option(critical_step)

                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                        f"critical_step={step_idx}, chosen={chosen_code}"
                    )

                suggested_code = None
                if feasible_action_codes:
                    improvement_prompt = build_step_improvement_prompt(
                        state_text=critical_step["state_text"],
                        candidate_action_text=f"{chosen_code}",
                        feasible_jobs=feasible_action_codes,
                        reflection_memory=reflection_memory,
                        step_diagnostics=_build_step_diagnostics(critical_step),
                    )
                    improvement_prompt = (
                        f"{improvement_prompt}\n"
                        f"Episode summary: final makespan={best_makespan}.\n"
                        f"Target step={step_idx}, chosen={chosen_code}.\n"
                        "Select an action that reduces bottleneck risk and final makespan."
                    )
                    reflection_prompt = _build_step_improvement_chat_prompt(
                        tokenizer, improvement_prompt
                    )
                    try:
                        suggested_code, _, _, _ = generate_step_action(
                            model=model,
                            tokenizer=tokenizer,
                            prompt_text=reflection_prompt,
                            feasible_action_codes=feasible_action_codes,
                            device=device,
                            max_new_tokens=step_action_max_new_tokens,
                            temperature=temperature,
                            top_p=top_p,
                            top_k=top_k,
                            use_masking=use_masking,
                            action_code_width=action_code_width,
                            action_code_cap=action_code_cap,
                        )
                    except StepActionParseError as exc:
                        improvement_notes.append(
                            f"pass {pass_idx + 1} step {step_idx}: improvement parse failed ({exc})"
                        )
                        if print_step_trace:
                            print(
                                f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                                f"step {step_idx}: parse failed ({exc})"
                            )

                if (not suggested_code or str(suggested_code) == chosen_code) and best_alt is not None:
                    alt_code = str(best_alt["action_code"])
                    if alt_code in action_code_to_job and alt_code != chosen_code:
                        suggested_code = alt_code
                        improvement_notes.append(
                            f"pass {pass_idx + 1} step {step_idx}: deterministic critic suggested {alt_code}"
                        )
                        if print_step_trace:
                            print(
                                f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                                f"step {step_idx}: critic suggested {alt_code}"
                            )

                if not suggested_code:
                    continue
                suggested_job = int(action_code_to_job[suggested_code])
                if suggested_job == chosen_job:
                    continue
                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)} "
                        f"step {step_idx}: suggested={suggested_code} -> Job {suggested_job}"
                    )
                guidance_map[step_idx] = {
                    "preferred_job": int(suggested_job),
                    "avoid_jobs": [int(chosen_job)],
                    "reason": (
                        f"chosen={chosen_code}/Job{chosen_job} showed high bottleneck risk; "
                        f"prefer {suggested_code}/Job{suggested_job}"
                    ),
                    "suggested_action_code": str(suggested_code),
                }

            if not guidance_map:
                improvement_notes.append(f"pass {pass_idx + 1}: no actionable guidance generated")
                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                        "no actionable guidance generated"
                    )
                continue

            cand_ms, cand_feasible, cand_traces, cand_steps, _ = _rollout_once(
                code_seed=int(action_code_seed) + (pass_idx + 1) * 9973,
                guidance_by_step=guidance_map,
                reflection_memory_text=reflection_memory,
            )
            if cand_feasible and math.isfinite(cand_ms) and cand_ms < best_makespan:
                improvement_notes.append(
                    f"pass {pass_idx + 1}: improved {best_makespan:.1f} -> {cand_ms:.1f} "
                    f"with guided_steps={sorted(guidance_map.keys())}"
                )
                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                        f"improved {best_makespan:.1f} -> {cand_ms:.1f}"
                    )
                best_makespan = float(cand_ms)
                best_traces = cand_traces
                best_step_records = cand_steps
            else:
                improvement_notes.append(
                    f"pass {pass_idx + 1}: no improvement ({best_makespan:.1f} vs {float(cand_ms):.1f})"
                )
                if print_step_trace:
                    print(
                        f"[Episode Improvement] pass {pass_idx + 1}/{int(step_reflection_passes)}: "
                        f"no improvement ({best_makespan:.1f} vs {float(cand_ms):.1f})"
                    )

    if print_step_trace:
        _print_step_trace(best_step_records)
        if improvement_notes:
            print("episode_improvement_notes:")
            for note in improvement_notes:
                print(" -", note)

    return float(best_makespan), True, best_traces

def compute_log_prob_mean(
    model,
    sequence_ids: torch.Tensor,
    prompt_len: int,
    device: torch.device,
    require_grad: bool,
) -> torch.Tensor:
    """
    Compute mean log-probability over generated tokens only.
    """
    seq = sequence_ids.unsqueeze(0).to(device)
    input_ids = seq[:, :-1]
    labels = seq[:, 1:].clone()
    labels[:, : max(prompt_len - 1, 0)] = -100

    if require_grad:
        outputs = model(input_ids=input_ids, labels=labels)
    else:
        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=labels)

    token_mask = (labels != -100)
    token_count = int(token_mask.sum().item())
    if token_count == 0:
        return torch.tensor(0.0, device=device)

    return -outputs.loss

def reinforce_step(
    batch: EpisodeBatch,
    optimizer: AdamW,
) -> float:
    """Perform a single REINFORCE update over collected samples."""

    advantages = batch.advantages
    if advantages.numel() > 1:
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    advantages = advantages.detach()
    loss = -(batch.log_probs * advantages).sum()

    optimizer.zero_grad()
    loss.backward()
    params = []
    for group in optimizer.param_groups:
        params.extend(group["params"])
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()

    return loss.item()

def grpo_step(
    samples: List[TrajectorySample],
    model,
    optimizer: AdamW,
    device: torch.device,
    clip_epsilon: float = 0.2,
    grpo_epochs: int = 1,
    kl_coef: float = 0.0,
) -> Tuple[float, float]:
    """
    PPO-style clipped policy update with group-normalized advantages.
    """
    if not samples:
        return 0.0, 0.0

    old_log_probs = torch.stack([s.old_log_prob for s in samples]).to(device).detach()
    advantages = torch.tensor([s.advantage for s in samples], dtype=torch.float32, device=device)

    total_loss = 0.0
    total_kl = 0.0
    for _ in range(grpo_epochs):
        current_log_probs = []
        for s in samples:
            log_prob = compute_log_prob_mean(
                model=model,
                sequence_ids=s.sequence_ids,
                prompt_len=s.prompt_len,
                device=device,
                require_grad=True,
            )
            current_log_probs.append(log_prob)
        current_log_probs = torch.stack(current_log_probs)

        log_ratio = current_log_probs - old_log_probs
        ratio = torch.exp(torch.clamp(log_ratio, -20, 20))
        unclipped_obj = ratio * advantages
        clipped_obj = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * advantages
        policy_loss = -torch.mean(torch.min(unclipped_obj, clipped_obj))

        approx_kl = torch.mean((ratio - 1.0) - torch.log(ratio + 1e-8))
        loss = policy_loss + kl_coef * approx_kl

        optimizer.zero_grad()
        loss.backward()
        params = []
        for group in optimizer.param_groups:
            params.extend(group["params"])
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        total_kl += float(approx_kl.item())

    return total_loss / grpo_epochs, total_kl / grpo_epochs

def build_bopo_step_pairs(
    group_rollouts: List[Dict],
    rng: np.random.Generator,
    min_relative_gap: float = 0.0,
    max_pairs_per_group: int = 256,
    max_step_pairs_per_pair: int = 32,
    pair_mode: str = "shared_prefix",
) -> List[BOPOStepPair]:
    """
    Build BOPO preference pairs from a rollout group.

    Strategy:
      1. Keep feasible rollouts only.
      2. Sort by makespan (smaller is better), anchor best rollout.
      3. Pair best vs each loser if relative gap >= threshold.
      4. For each episode pair, either:
         - sample aligned step indices (`aligned`)
         - or keep only the first divergence step after shared prefix (`shared_prefix`)
    """
    feasible_rollouts = [
        r
        for r in group_rollouts
        if bool(r.get("feasible"))
        and math.isfinite(float(r.get("makespan", float("inf"))))
        and len(r.get("traces", [])) > 0
    ]
    if len(feasible_rollouts) < 2:
        return []

    feasible_rollouts.sort(key=lambda x: float(x["makespan"]))
    winner = feasible_rollouts[0]
    winner_ms = float(winner["makespan"])
    winner_traces = winner["traces"]

    resolved_pair_mode = str(pair_mode).strip().lower()
    if resolved_pair_mode not in {"aligned", "shared_prefix"}:
        raise ValueError(f"Unsupported pair_mode={pair_mode}")

    def _first_divergence_index(w_traces, l_traces):
        n_steps_local = min(len(w_traces), len(l_traces))
        for idx in range(n_steps_local):
            if int(w_traces[idx].chosen_job) != int(l_traces[idx].chosen_job):
                return idx
        return None

    pairs: List[BOPOStepPair] = []
    for loser in feasible_rollouts[1:]:
        loser_ms = float(loser["makespan"])
        if loser_ms <= winner_ms:
            continue
        rel_gap = (loser_ms - winner_ms) / max(loser_ms, 1.0)
        if rel_gap < float(min_relative_gap):
            continue

        loser_traces = loser["traces"]
        n_steps = min(len(winner_traces), len(loser_traces))
        if n_steps <= 0:
            continue

        if resolved_pair_mode == "shared_prefix":
            div_idx = _first_divergence_index(winner_traces, loser_traces)
            if div_idx is None:
                continue
            indices = [int(div_idx)]
        else:
            indices = list(range(n_steps))
            max_steps = int(max_step_pairs_per_pair)
            if max_steps > 0 and n_steps > max_steps:
                picked = rng.choice(n_steps, size=max_steps, replace=False)
                indices = [int(x) for x in picked.tolist()]

        for step_idx in indices:
            w_trace = winner_traces[step_idx]
            l_trace = loser_traces[step_idx]
            pairs.append(
                BOPOStepPair(
                    winner_sequence_ids=w_trace.sequence_ids,
                    winner_prompt_len=int(w_trace.prompt_len),
                    loser_sequence_ids=l_trace.sequence_ids,
                    loser_prompt_len=int(l_trace.prompt_len),
                    relative_gap=float(rel_gap),
                    winner_makespan=float(winner_ms),
                    loser_makespan=float(loser_ms),
                )
            )

    max_pairs = int(max_pairs_per_group)
    if max_pairs > 0 and len(pairs) > max_pairs:
        picked = rng.choice(len(pairs), size=max_pairs, replace=False)
        pairs = [pairs[int(i)] for i in picked.tolist()]

    return pairs

def bopo_step(
    pairs: List[BOPOStepPair],
    model,
    optimizer: AdamW,
    device: torch.device,
    beta: float = 2.0,
    gap_scale: float = 3.0,
    margin: float = 0.0,
) -> Tuple[float, float, int]:
    """
    BOPO-style pairwise preference optimization.

    loss = -log sigma( beta * (1 + gap_scale * rel_gap) * (logp_win - logp_lose - margin) )
    """
    if not pairs:
        return 0.0, 0.0, 0

    total_loss = 0.0
    total_gap = 0.0
    updates = 0

    for p in pairs:
        try:
            lp_w = compute_log_prob_mean(
                model=model,
                sequence_ids=p.winner_sequence_ids,
                prompt_len=p.winner_prompt_len,
                device=device,
                require_grad=True,
            )
            lp_l = compute_log_prob_mean(
                model=model,
                sequence_ids=p.loser_sequence_ids,
                prompt_len=p.loser_prompt_len,
                device=device,
                require_grad=True,
            )

            rel_gap = max(0.0, float(p.relative_gap))
            scaled_beta = float(beta) * (1.0 + float(gap_scale) * rel_gap)
            pref_logit = scaled_beta * (lp_w - lp_l - float(margin))
            loss = -F.logsigmoid(pref_logit)

            optimizer.zero_grad()
            loss.backward()
            params = []
            for group in optimizer.param_groups:
                params.extend(group["params"])
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            total_loss += float(loss.item())
            total_gap += rel_gap
            updates += 1

            del lp_w, lp_l, loss, pref_logit
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                optimizer.zero_grad(set_to_none=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise

    return total_loss / max(updates, 1), total_gap / max(updates, 1), updates

def resolve_upload_dir(output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
    output_dir = Path(output_dir)
    source = str(upload_source or 'latest_checkpoint')
    if source == 'latest_checkpoint':
        checkpoints = sorted(
            output_dir.glob('checkpoint-*'),
            key=lambda p: p.name,
        )
        if not checkpoints:
            raise FileNotFoundError(f'no checkpoint-* directories found under {output_dir}')
        return checkpoints[-1]
    if source == 'final':
        p = output_dir / 'final'
        if not p.exists():
            raise FileNotFoundError(f'final folder not found: {p}')
        return p
    if checkpoint_tag:
        p = output_dir / str(checkpoint_tag)
        if not p.exists():
            raise FileNotFoundError(f'checkpoint folder not found: {p}')
        return p
    raise ValueError(f'Unsupported upload_source={upload_source!r}')


## 3) 비교 실행 (GRPO vs REINFORCE)


In [ ]:
import gc


def _set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _resolve_dataset_path(dataset_source: str, hf_repo: str, hf_file: str, local_path: str, token: str):
    if dataset_source == 'hf':
        return hf_hub_download(
            repo_id=hf_repo,
            repo_type='dataset',
            filename=hf_file,
            token=token,
        )
    if dataset_source == 'local':
        return os.path.expanduser(local_path)
    raise ValueError("dataset_source must be 'hf' or 'local'")


def _load_train_split(cfg):
    if cfg['use_random_problems']:
        return None
    if cfg.get('dataset_path'):
        dataset_path = os.path.expanduser(cfg['dataset_path'])
    else:
        dataset_path = _resolve_dataset_path(
            dataset_source=cfg['dataset_source'],
            hf_repo=cfg['dataset_hf_repo'],
            hf_file=cfg['dataset_hf_file'],
            local_path=cfg['dataset_local_path'],
            token=cfg['hf_token'],
        )
    print('train dataset path:', dataset_path)
    ds = load_dataset('json', data_files=dataset_path)
    return ds['train']


def _load_eval_split(cfg, train_split_fallback=None):
    mode = cfg.get('compare_eval_source', 'same_as_train')
    if mode == 'same_as_train':
        if train_split_fallback is None:
            raise ValueError('compare_eval_source=same_as_train but train split is None')
        return train_split_fallback
    if mode == 'hf':
        eval_path = _resolve_dataset_path(
            dataset_source='hf',
            hf_repo=cfg['compare_eval_hf_repo'],
            hf_file=cfg['compare_eval_hf_file'],
            local_path=cfg['compare_eval_local_path'],
            token=cfg['hf_token'],
        )
    elif mode == 'local':
        eval_path = os.path.expanduser(cfg['compare_eval_local_path'])
    else:
        raise ValueError("compare_eval_source must be one of {'same_as_train','hf','local'}")
    print('eval dataset path:', eval_path)
    ds = load_dataset('json', data_files=eval_path)
    return ds['train']


def _summarize_trainable_parameters(model):
    trainable = 0
    total = 0
    for param in model.parameters():
        n = int(param.numel())
        total += n
        if param.requires_grad:
            trainable += n
    ratio = (float(trainable) / float(total)) if total > 0 else 0.0
    return trainable, total, ratio


def _validate_rl_update_mode(model, update_mode: str, model_path: str):
    trainable, total, ratio = _summarize_trainable_parameters(model)
    print(f'[Info] RL trainable params: {trainable:,} / {total:,} ({ratio * 100:.2f}%)')
    if update_mode == 'adapter_only':
        if trainable <= 0:
            raise ValueError(f'RL update mode is adapter_only, but no trainable parameters were found. model_path={model_path!r}')
    elif update_mode == 'full':
        if trainable <= 0:
            raise ValueError(f'RL update mode is full, but no trainable parameters were found. model_path={model_path!r}')
    else:
        raise ValueError(f'Unsupported rl_update_mode={update_mode}')
    return trainable, total, ratio


def train_one_algorithm(algo_name: str, cfg_base: dict):
    cfg = dict(cfg_base)
    cfg['rl_algo'] = str(algo_name)

    run_dir = Path(cfg['compare_output_dir']) / f"algo_{algo_name}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg['output_dir'] = str(run_dir)

    _set_all_seeds(int(cfg['seed']))

    model_path = cfg['model_path'] if cfg['model_path'] else MODEL_MAP[cfg['model_type']]
    print(f"\n[train:{algo_name}] loading model: {model_path}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=cfg['max_seq_length'],
        load_in_4bit=cfg['load_in_4bit'],
        dtype=torch.bfloat16 if cfg['dtype'] == 'bfloat16' else torch.float16,
        local_files_only=False,
    )
    if hasattr(model, 'for_training'):
        model.for_training()
    model.to(device)
    model.train()

    optimizer = AdamW(model.parameters(), lr=cfg['learning_rate'])
    rng = np.random.default_rng(int(cfg['seed']))
    train_split = _load_train_split(cfg)
    baseline_tracker = ExponentialBaseline(beta=cfg['baseline_beta'])

    history = []
    for epoch in range(int(cfg['epochs'])):
        grpo_samples = []
        bopo_pairs_epoch = []
        reinforce_updates = 0
        reinforce_loss_sum = 0.0
        bopo_updates = 0
        bopo_loss_total = 0.0
        bopo_gap_total = 0.0
        epoch_rewards = []
        epoch_makespans = []
        epoch_feasible = 0
        epoch_rollouts = 0

        with trange(int(cfg['episodes_per_epoch']), desc=f"[{algo_name}] Epoch {epoch+1}/{cfg['epochs']}") as t:
            for _ in t:
                if cfg['use_random_problems']:
                    instance = generate_random_instance(
                        num_jobs=cfg['random_jobs'],
                        num_machines=cfg['random_machines'],
                        process_time_range=(cfg['random_time_low'], cfg['random_time_high']),
                        rng=rng,
                    )
                    inst = instance['inst_for_ortools']
                else:
                    example = random.choice(train_split)
                    _, _, inst = parse_matrix_form_jssp(example['matrix'])

                if cfg['rl_algo'] == 'grpo':
                    group_rollouts = []
                    rewards = []
                    for _ in range(max(1, int(cfg['group_size']))):
                        rollout_cfg = dict(cfg)
                        rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                        try:
                            makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                        except StepActionParseError as exc:
                            makespan = float(cfg['invalid_makespan_penalty'])
                            feasible = False
                            traces = []
                            if cfg.get('print_step_trace', False):
                                print(f'[warn:{algo_name}] GRPO rollout parse failed -> penalty applied: {exc}')

                        reward = -makespan if feasible and math.isfinite(makespan) else -float(cfg['invalid_makespan_penalty'])
                        epoch_rollouts += 1
                        if feasible and math.isfinite(makespan):
                            epoch_feasible += 1
                            epoch_makespans.append(float(makespan))
                        epoch_rewards.append(float(reward))

                        step_samples = []
                        for tr in traces:
                            old_lp = compute_log_prob_mean(
                                model=model,
                                sequence_ids=tr.sequence_ids,
                                prompt_len=tr.prompt_len,
                                device=device,
                                require_grad=False,
                            ).detach()
                            step_samples.append({
                                'sequence_ids': tr.sequence_ids,
                                'prompt_len': tr.prompt_len,
                                'old_log_prob': old_lp,
                            })

                        group_rollouts.append(
                            {
                                'reward': float(reward),
                                'feasible': bool(feasible),
                                'makespan': float(makespan),
                                'step_samples': step_samples,
                            }
                        )
                        rewards.append(float(reward))

                    rewards_t = torch.tensor(rewards, dtype=torch.float32)
                    mean_r = float(rewards_t.mean().item())
                    std_r = float(rewards_t.std(unbiased=False).item())
                    denom = std_r + 1e-8

                    for rollout in group_rollouts:
                        advantage = (rollout['reward'] - mean_r) / denom
                        for s in rollout['step_samples']:
                            grpo_samples.append(
                                TrajectorySample(
                                    sequence_ids=s['sequence_ids'],
                                    prompt_len=s['prompt_len'],
                                    reward=rollout['reward'],
                                    advantage=float(advantage),
                                    old_log_prob=s['old_log_prob'],
                                    feasible=rollout['feasible'],
                                    makespan=rollout['makespan'],
                                )
                            )

                    feasible_makespans = [r['makespan'] for r in group_rollouts if r['feasible']]
                    best_ms = min(feasible_makespans) if feasible_makespans else float(cfg['invalid_makespan_penalty'])
                    t.set_postfix(algo='grpo', best_makespan=f'{best_ms:.1f}', group_reward=f'{mean_r:.1f}')

                elif cfg['rl_algo'] == 'bopo':
                    group_rollouts = []
                    for _ in range(max(2, int(cfg['group_size']))):
                        rollout_cfg = dict(cfg)
                        rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                        try:
                            makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                        except StepActionParseError as exc:
                            makespan = float(cfg['invalid_makespan_penalty'])
                            feasible = False
                            traces = []
                            if cfg.get('print_step_trace', False):
                                print(f'[warn:{algo_name}] BOPO rollout parse failed -> penalty applied: {exc}')

                        reward = -makespan if feasible and math.isfinite(makespan) else -float(cfg['invalid_makespan_penalty'])
                        epoch_rollouts += 1
                        epoch_rewards.append(float(reward))
                        if feasible and math.isfinite(makespan):
                            epoch_feasible += 1
                            epoch_makespans.append(float(makespan))

                        group_rollouts.append(
                            {
                                'feasible': bool(feasible),
                                'makespan': float(makespan),
                                'traces': traces,
                            }
                        )

                    feasible_makespans = [r['makespan'] for r in group_rollouts if r['feasible']]
                    best_ms = min(feasible_makespans) if feasible_makespans else float(cfg['invalid_makespan_penalty'])

                    bopo_pairs = build_bopo_step_pairs(
                        group_rollouts=group_rollouts,
                        rng=rng,
                        min_relative_gap=cfg['bopo_min_relative_gap'],
                        max_pairs_per_group=cfg['bopo_max_pairs_per_group'],
                        max_step_pairs_per_pair=cfg['bopo_max_step_pairs_per_pair'],
                    )
                    if not bopo_pairs:
                        t.set_postfix(algo='bopo', best_makespan=f'{best_ms:.1f}', pairs=0)
                        continue

                    bopo_pairs_epoch.extend(bopo_pairs)

                    t.set_postfix(
                        algo='bopo-collect',
                        best_makespan=f'{best_ms:.1f}',
                        pairs=len(bopo_pairs),
                        epoch_pairs=len(bopo_pairs_epoch),
                    )

                else:
                    rollout_cfg = dict(cfg)
                    rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                    try:
                        makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                    except StepActionParseError as exc:
                        epoch_rollouts += 1
                        epoch_rewards.append(-float(cfg['invalid_makespan_penalty']))
                        if cfg.get('print_step_trace', False):
                            print(f'[warn:{algo_name}] REINFORCE rollout parse failed -> skip: {exc}')
                        t.set_postfix_str('parse-failed rollout, skipping')
                        continue
                    if not feasible or not traces or not math.isfinite(makespan):
                        epoch_rollouts += 1
                        epoch_rewards.append(-float(cfg['invalid_makespan_penalty']))
                        t.set_postfix_str('invalid rollout, skipping')
                        continue

                    epoch_rollouts += 1
                    epoch_feasible += 1
                    epoch_makespans.append(float(makespan))

                    reward = -makespan
                    epoch_rewards.append(float(reward))
                    _, mwkr_makespan = mwkr_schedule(inst)
                    baseline = -mwkr_makespan if math.isfinite(mwkr_makespan) else reward
                    if cfg['use_running_baseline']:
                        baseline = baseline_tracker.update(reward)
                    advantage = reward - baseline

                    adv_tensor = torch.tensor(float(advantage), device=device)
                    trace_update_count = 0
                    trace_loss_sum = 0.0
                    for tr in traces:
                        lp = compute_log_prob_mean(
                            model=model,
                            sequence_ids=tr.sequence_ids,
                            prompt_len=tr.prompt_len,
                            device=device,
                            require_grad=True,
                        )
                        loss = -(lp * adv_tensor)
                        optimizer.zero_grad()
                        loss.backward()
                        params = []
                        for g in optimizer.param_groups:
                            params.extend(g['params'])
                        torch.nn.utils.clip_grad_norm_(params, 1.0)
                        optimizer.step()

                        trace_update_count += 1
                        trace_loss_sum += float(loss.item())
                        del lp, loss

                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    reinforce_updates += int(trace_update_count)
                    reinforce_loss_sum += float(trace_loss_sum)
                    avg_trace_loss = (trace_loss_sum / trace_update_count) if trace_update_count > 0 else 0.0

                    t.set_postfix(
                        algo='reinforce',
                        makespan=f'{makespan:.1f}',
                        reward=f'{reward:.1f}',
                        advantage=f'{advantage:.1f}',
                        steps=len(traces),
                        trace_updates=trace_update_count,
                        loss=f'{avg_trace_loss:.4f}',
                    )

        if cfg['rl_algo'] == 'grpo':
            if not grpo_samples:
                print(f'[train:{algo_name}] epoch {epoch+1}: no GRPO samples; skipping update')
                loss_value, approx_kl = None, None
            else:
                loss_value, approx_kl = grpo_step(
                    samples=grpo_samples,
                    model=model,
                    optimizer=optimizer,
                    device=device,
                    clip_epsilon=cfg['clip_epsilon'],
                    grpo_epochs=cfg['grpo_epochs'],
                    kl_coef=cfg['kl_coef'],
                )
                print(f"[train:{algo_name}] epoch {epoch+1}: GRPO loss={loss_value:.6f}, approx_kl={approx_kl:.6f}, samples={len(grpo_samples)}")
        elif cfg['rl_algo'] == 'bopo':
            approx_kl = None
            if bopo_updates <= 0:
                print(f'[train:{algo_name}] epoch {epoch+1}: no valid BOPO preference pairs; skipping update')
                loss_value = None
            else:
                avg_pair_loss = bopo_loss_total / max(bopo_updates, 1)
                avg_gap = bopo_gap_total / max(bopo_updates, 1)
                loss_value = avg_pair_loss
                print(
                    f"[train:{algo_name}] epoch {epoch+1}: "
                    f"BOPO pair-updates={bopo_updates}, avg_pair_loss={avg_pair_loss:.6f}, avg_rel_gap={avg_gap:.6f}"
                )
        else:
            approx_kl = None
            if reinforce_updates <= 0:
                print(f'[train:{algo_name}] epoch {epoch+1}: no valid episodes; skipping update')
                loss_value = None
            else:
                avg_loss = reinforce_loss_sum / max(reinforce_updates, 1)
                loss_value = avg_loss
                print(
                    f"[train:{algo_name}] epoch {epoch+1}: "
                    f"REINFORCE trace-updates={reinforce_updates}, avg_trace_loss={avg_loss:.6f}"
                )

        if cfg['save_every_epoch']:
            ckpt = run_dir / f'checkpoint-epoch-{epoch+1}'
            ckpt.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(ckpt))
            tokenizer.save_pretrained(str(ckpt))
            print('[saved]', ckpt)

        hist = {
            'algo': algo_name,
            'epoch': epoch + 1,
            'loss': loss_value,
            'approx_kl': approx_kl,
            'mean_reward': float(np.mean(epoch_rewards)) if epoch_rewards else None,
            'mean_makespan_feasible': float(np.mean(epoch_makespans)) if epoch_makespans else None,
            'feasible_rate': float(epoch_feasible / epoch_rollouts) if epoch_rollouts else 0.0,
            'num_rollouts': int(epoch_rollouts),
        }
        history.append(hist)

    final_dir = run_dir / 'final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    print(f"[train:{algo_name}] done. final_dir={final_dir}")
    return model, tokenizer, run_dir, final_dir, history, train_split


def evaluate_algorithm(model, tokenizer, eval_split, cfg_base, algo_name):
    cfg = dict(cfg_base)
    eval_instances = min(int(cfg.get('compare_eval_instances', 20)), len(eval_split))
    sample_count = max(1, int(cfg.get('compare_eval_num_return_sequences', 1)))

    rows = []
    for idx in trange(eval_instances, desc=f"[eval:{algo_name}]"):
        example = eval_split[idx]
        n, m, inst, real_ms = parse_matrix_form_jssp(example['matrix'])
        benchmark = real_ms if real_ms is not None else example.get('makespan')

        sample_feasible = []
        sample_makespans = []
        feasible_values = []

        for s in range(sample_count):
            rollout_cfg = dict(cfg)
            rollout_cfg['print_step_trace'] = False
            rollout_cfg['enable_step_improvement'] = bool(cfg.get('compare_eval_enable_step_improvement', False))
            rollout_cfg['action_code_seed'] = int(cfg['seed']) + idx * 10007 + s * 997
            makespan, feasible, _ = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
            is_ok = bool(feasible and math.isfinite(makespan))
            sample_feasible.append(is_ok)
            if is_ok:
                ms = float(makespan)
                sample_makespans.append(ms)
                feasible_values.append(ms)
            else:
                sample_makespans.append(None)

        best_makespan = min(feasible_values) if feasible_values else None
        mean_ms = float(np.mean(feasible_values)) if feasible_values else None
        var_ms = float(np.var(feasible_values)) if feasible_values else None
        std_ms = float(np.std(feasible_values)) if feasible_values else None
        gap = None
        if best_makespan is not None and benchmark not in (None, 0):
            gap = abs((best_makespan - float(benchmark)) / float(benchmark))

        row = {
            'algo': algo_name,
            'index': int(idx),
            'num_jobs': int(n),
            'num_machines': int(m),
            'benchmark_makespan': benchmark,
            'best_makespan': best_makespan,
            'mean_makespan': mean_ms,
            'var_makespan': var_ms,
            'std_makespan': std_ms,
            'num_attempts': int(sample_count),
            'num_feasible': int(len(feasible_values)),
            'feasible_rate': float(len(feasible_values) / sample_count),
            'path': example.get('path', ''),
            'gap': gap,
        }
        for k in range(sample_count):
            row[f'sample{k+1}_feasible'] = int(bool(sample_feasible[k]))
            row[f'sample{k+1}_makespan'] = sample_makespans[k]
        rows.append(row)

    gaps = [r['gap'] for r in rows if r.get('gap') is not None]
    bests = [r['best_makespan'] for r in rows if r.get('best_makespan') is not None]
    summary = {
        'algo': algo_name,
        'num_eval_instances': int(eval_instances),
        'avg_gap': float(np.mean(gaps)) if gaps else None,
        'std_gap': float(np.std(gaps)) if gaps else None,
        'avg_best_makespan': float(np.mean(bests)) if bests else None,
        'feasible_problem_rate': float(sum(1 for r in rows if r['num_feasible'] > 0) / len(rows)) if rows else 0.0,
    }
    return summary, rows


def write_dict_csv(path: Path, rows: list):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        print('no rows to save:', path)
        return
    fieldnames = sorted({k for r in rows for k in r.keys()})
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)
    print('saved:', path)


# ---------------------- compare run ----------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if CFG.get('hf_token'):
    login(token=CFG['hf_token'], add_to_git_credential=False)

compare_dir = Path(CFG['compare_output_dir'])
compare_dir.mkdir(parents=True, exist_ok=True)

algorithms = [str(a).strip().lower() for a in CFG.get('compare_algorithms', ['grpo', 'reinforce', 'bopo'])]
algorithms = [a for a in algorithms if a in {'grpo', 'reinforce', 'bopo'}]
if not algorithms:
    raise ValueError("compare_algorithms must contain at least one of {'grpo','reinforce','bopo'}")

print('compare algorithms:', algorithms)
print('seed:', CFG['seed'])

all_summary_rows = []
all_problem_rows = []
all_train_rows = []

for algo in algorithms:
    model, tokenizer, run_dir, final_dir, train_history, train_split = train_one_algorithm(algo, CFG)
    all_train_rows.extend(train_history)

    eval_split = _load_eval_split(CFG, train_split_fallback=train_split)
    summary, rows = evaluate_algorithm(model, tokenizer, eval_split, CFG, algo)
    summary['run_dir'] = str(run_dir)
    summary['final_dir'] = str(final_dir)

    all_summary_rows.append(summary)
    all_problem_rows.extend(rows)

    # cleanup before next algorithm
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_csv = compare_dir / 'compare_summary.csv'
problems_csv = compare_dir / 'compare_per_problem.csv'
train_csv = compare_dir / 'compare_train_history.csv'
write_dict_csv(summary_csv, all_summary_rows)
write_dict_csv(problems_csv, all_problem_rows)
write_dict_csv(train_csv, all_train_rows)

print('\n=== Compare Summary ===')
for row in all_summary_rows:
    print(row)

print('compare done')
print(' - summary:', summary_csv)
print(' - per-problem:', problems_csv)
print(' - train-history:', train_csv)
